**向量检索**

In [9]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings.dashscope import DashScopeEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("./langchain.txt", encoding="utf-8")
pages = loader.load()
# print(pages[:3])
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
)

chunks = text_splitter.split_documents(pages)
# print(chunks)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = vector_retriever.invoke("BM25 算法")
print(results)

[Document(metadata={'source': './langchain.txt'}, page_content='## 常见问题\n\nQ: LangChain 1.0 有什么新特性？\nA: 更简洁的 API，内置 LangGraph，改进的中间件系统\n\nQ: 如何选择向量数据库？\nA: Pinecone 适合生产，Chroma 适合开发，FAISS 适合离线\n\nQ: BM25 是什么？\nA: Best Match 25，一种基于词频的检索算法，适合精确匹配'), Document(metadata={'source': './langchain.txt'}, page_content='## 常见问题\n\nQ: LangChain 1.0 有什么新特性？\nA: 更简洁的 API，内置 LangGraph，改进的中间件系统\n\nQ: 如何选择向量数据库？\nA: Pinecone 适合生产，Chroma 适合开发，FAISS 适合离线\n\nQ: BM25 是什么？\nA: Best Match 25，一种基于词频的检索算法，适合精确匹配'), Document(metadata={'source': './langchain.txt'}, page_content='## 常见问题\n\nQ: LangChain 1.0 有什么新特性？\nA: 更简洁的 API，内置 LangGraph，改进的中间件系统\n\nQ: 如何选择向量数据库？\nA: Pinecone 适合生产，Chroma 适合开发，FAISS 适合离线\n\nQ: BM25 是什么？\nA: Best Match 25，一种基于词频的检索算法，适合精确匹配')]


**BM25检索**

pip install rank_bm25

In [8]:
from langchain_community.retrievers import BM25Retriever


bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

print(f"\n[OK] BM25 检索器已创建")
print(f"  检索数量: k=3")

# 测试查询
results = bm25_retriever.invoke("BM25 算法")
print(results)
print(f"结果数: {len(results)}")


[OK] BM25 检索器已创建
  检索数量: k=3
[Document(metadata={'source': './langchain.txt'}, page_content='## 常见问题\n\nQ: LangChain 1.0 有什么新特性？\nA: 更简洁的 API，内置 LangGraph，改进的中间件系统\n\nQ: 如何选择向量数据库？\nA: Pinecone 适合生产，Chroma 适合开发，FAISS 适合离线\n\nQ: BM25 是什么？\nA: Best Match 25，一种基于词频的检索算法，适合精确匹配'), Document(metadata={'source': './langchain.txt'}, page_content='agent = create_agent(\n    model=model,\n    tools=[search_docs],\n    system_prompt="你是助手"\n)\n```\n\n## 性能优化\n\n1. Chunk 大小：建议 500-1000 字符\n2. Chunk 重叠：10-20%\n3. 检索数量：k=3-5\n4. 混合搜索权重：向量 0.6, BM25 0.4'), Document(metadata={'source': './langchain.txt'}, page_content='## 代码示例\n\n```python\nfrom langchain.agents import create_agent\nfrom langchain_core.tools import tool\n\n@tool\ndef search_docs(query: str) -> str:\n    """搜索文档"""\n    return "结果"')]
结果数: 3


**混合检索器**

In [15]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[1, 0]  # 稍微偏向向量搜索
)

print(f"\n[OK] 混合检索器已创建")
print(f"  组合: BM25 (40%) + Vector (60%)")
print(f"  算法: RRF (Reciprocal Rank Fusion)")


# 对比测试
test_queries = [
    ("语义查询", "LangChain 的主要功能是什么？"),
    ("精确匹配", "langchain>=1.0.0"),
    ("混合查询", "BM25 算法如何工作？"),
]

print(f"\n对比测试:")
for query_type, query in test_queries:
    print(f"\n  [{query_type}] {query}")

    # BM25 结果
    bm25_results = bm25_retriever.invoke(query)
    bm25_preview = bm25_results[0].page_content[:90].replace("\n", " ") if bm25_results else "无"

    # 向量结果
    vector_results = vector_retriever.invoke(query)
    vector_preview = vector_results[0].page_content[:90].replace("\n", " ") if vector_results else "无"

    # 混合结果
    ensemble_results = ensemble_retriever.invoke(query)
    ensemble_preview = ensemble_results[0].page_content[:90].replace("\n", " ") if ensemble_results else "无"

    print(f"BM25: {bm25_preview}...")
    print(f"Vector: {vector_preview}...")
    print(f"Hybrid: {ensemble_preview}...")


[OK] 混合检索器已创建
  组合: BM25 (40%) + Vector (60%)
  算法: RRF (Reciprocal Rank Fusion)

对比测试:

  [语义查询] LangChain 的主要功能是什么？
BM25: # LangChain 框架详解  ## 核心组件  LangChain 提供以下核心组件：  1. Models (模型接口)    - 支持 OpenAI GPT-4, GPT...
Vector: # LangChain 框架详解  ## 核心组件  LangChain 提供以下核心组件：  1. Models (模型接口)    - 支持 OpenAI GPT-4, GPT...
Hybrid: # LangChain 框架详解  ## 核心组件  LangChain 提供以下核心组件：  1. Models (模型接口)    - 支持 OpenAI GPT-4, GPT...

  [精确匹配] langchain>=1.0.0
BM25: ## 常见问题  Q: LangChain 1.0 有什么新特性？ A: 更简洁的 API，内置 LangGraph，改进的中间件系统  Q: 如何选择向量数据库？ A: Pine...
Vector: ## 常见问题  Q: LangChain 1.0 有什么新特性？ A: 更简洁的 API，内置 LangGraph，改进的中间件系统  Q: 如何选择向量数据库？ A: Pine...
Hybrid: ## 常见问题  Q: LangChain 1.0 有什么新特性？ A: 更简洁的 API，内置 LangGraph，改进的中间件系统  Q: 如何选择向量数据库？ A: Pine...

  [混合查询] BM25 算法如何工作？
BM25: ## 常见问题  Q: LangChain 1.0 有什么新特性？ A: 更简洁的 API，内置 LangGraph，改进的中间件系统  Q: 如何选择向量数据库？ A: Pine...
Vector: ## 常见问题  Q: LangChain 1.0 有什么新特性？ A: 更简洁的 API，内置 LangGraph，改进的中间件系统  Q: 如何选择向量数据库？ A: Pine...
Hybrid: ## 常见问题  Q: L

**RAG Agent with Hybrid Search**

In [5]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


# 创建检索工具
@tool
def search_knowledge_base(query: str) -> str:
    """在知识库中搜索相关信息（混合检索）"""
    docs = ensemble_retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs[:2]])  # 只取前2个

print(f"[OK] 工具已创建: search_knowledge_base")

model = init_chat_model(model="deepseek:deepseek-chat")

# 创建Agent
agent = create_agent(
    model=model,
    tools=[search_knowledge_base],
    system_prompt="""
        你是一个有用的助手。
        使用 search_knowledge_base 工具搜索相关信息，然后回答问题。
        
        注意：
        1. 优先使用检索到的信息
        2. 如果信息不足，诚实告知
        3. 回答要简洁准确
    """
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "LangChain 有哪些核心组件?"}]
})

print(f"回答: {response['messages'][-1].content}")

[OK] 工具已创建: search_knowledge_base
回答: 基于我检索到的信息，LangChain的核心组件主要包括：

## LangChain 核心组件

### 1. Models (模型接口)
这是LangChain最核心的组件之一，支持多种大型语言模型：
- **OpenAI模型**：GPT-4, GPT-3.5
- **Anthropic Claude** 模型
- **Groq Llama** 模型
- 版本要求：langchain>=1.0.0

### 2. 其他重要组件（基于LangChain标准架构）
虽然搜索结果中只详细列出了Models组件，但根据LangChain的标准架构，通常还包括以下核心组件：

**Prompts (提示模板)**：
- 用于管理和优化提示词
- 支持模板化、变量替换

**Chains (链式调用)**：
- 将多个组件连接起来形成工作流
- 支持顺序链、条件链等

**Agents (智能代理)**：
- 能够使用工具和思考的智能体
- 支持ReAct、Self-ask等模式

**Memory (记忆)**：
- 对话历史管理
- 短期和长期记忆存储

**Document Loaders (文档加载器)**：
- 从各种来源加载文档
- 支持PDF、网页、数据库等

**Vector Stores (向量存储)**：
- 用于存储和检索嵌入向量
- 支持Pinecone、Chroma、FAISS等

### LangChain 1.0 新特性
- 更简洁的API设计
- 内置LangGraph（用于构建复杂工作流）
- 改进的中间件系统

**注意**：检索到的信息主要聚焦于Models组件，其他组件信息相对有限。如果您需要了解特定组件的详细信息，我可以为您进一步搜索。
